# Scraping historical stock prices with Python

I wanted to practice turning an HTML table into a dataset I could actually work with. This project extracts historical prices for Netflix and Amazon, cleans the values, and compares a manual BeautifulSoup approach with pandas' table reader.

The pages are saved Yahoo Finance snapshots, so the results describe the dates in those files rather than current market prices. My focus here is the extraction process, not making investment recommendations.

## Project outline
1. Download the HTML pages.
2. Locate the price tables and extract their rows.
3. Convert dates, prices, and volume into useful data types.
4. Check both extraction methods agree and export CSV files.

## Setup

Install the packages from `requirements.txt` before running this notebook. I use Python's built-in HTML parser for BeautifulSoup and lxml for `pandas.read_html`.

In [ ]:
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

BASE_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/"
)
SOURCES = {
    "Netflix": BASE_URL + "netflix_data_webpage.html",
    "Amazon": BASE_URL + "amazon_data_webpage.html",
}
COLUMNS = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]

## 1. Download the pages

I fetch each page once and reuse its HTML for both methods. A timeout prevents an unresponsive server from hanging the notebook, and `raise_for_status()` makes a failed download visible.

In [ ]:
def download_html(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.text


pages = {company: download_html(url) for company, url in SOURCES.items()}
for company, html in pages.items():
    soup = BeautifulSoup(html, "html.parser")
    print(f"{company}: {soup.title.get_text(strip=True) if soup.title else 'No title'}")

## 2. Extract the historical price table

Instead of assuming the first table is the right one, I look for the seven price-history headers. I collect rows in a list and create one DataFrame at the end. Rows such as dividend or split notices do not have seven separate price fields, so I count and skip them.

In [ ]:
def find_price_table(html):
    soup = BeautifulSoup(html, "html.parser")
    for table in soup.find_all("table"):
        headers = [cell.get_text(" ", strip=True) for cell in table.find_all("th")]
        # Yahoo snapshots sometimes add footnote markers to price headers.
        headers = [header.replace("*", "").strip() for header in headers]
        if headers == COLUMNS:
            return table
    raise ValueError("Could not find a table with the expected price-history headers.")


def extract_prices(html):
    table = find_price_table(html)
    records = []
    skipped_rows = 0
    for row in table.find_all("tr"):
        fields = row.find_all("td")
        if not fields:
            continue
        if len(fields) != len(COLUMNS) or any(field.has_attr("colspan") for field in fields):
            skipped_rows += 1
            continue
        records.append([field.get_text(strip=True) for field in fields])
    if not records:
        raise ValueError("The price table contains no usable rows.")
    return pd.DataFrame(records, columns=COLUMNS), skipped_rows

## 3. Clean the values and check the dataset

HTML values arrive as text. I remove thousands separators, parse dates, and store volume as an integer. The checks below catch missing values, duplicate dates, negative volume, and inconsistent high/low prices. I sort from oldest to newest so the first and last rows have a clear chronological meaning. Unexpected values stop execution rather than silently becoming missing data.

In [ ]:
def clean_prices(frame):
    cleaned = frame.loc[:, COLUMNS].copy()
    cleaned["Date"] = pd.to_datetime(cleaned["Date"], format="%b %d, %Y")
    for column in COLUMNS[1:]:
        values = cleaned[column].astype(str).str.replace(",", "", regex=False)
        cleaned[column] = pd.to_numeric(values, errors="raise")
    if cleaned.empty or cleaned.isna().any().any():
        raise ValueError("Empty dataset or missing price-history values.")
    if cleaned["Date"].duplicated().any():
        raise ValueError("Duplicate dates found.")
    if (cleaned["Volume"] < 0).any() or (cleaned["Volume"] % 1 != 0).any():
        raise ValueError("Volume must be a non-negative integer.")
    cleaned["Volume"] = cleaned["Volume"].astype("int64")
    if (cleaned[COLUMNS[1:6]] <= 0).any().any():
        raise ValueError("Prices must be positive.")
    if (cleaned["High"] < cleaned[["Open", "Low", "Close"]].max(axis=1)).any():
        raise ValueError("High price is below another price in the same row.")
    if (cleaned["Low"] > cleaned[["Open", "High", "Close"]].min(axis=1)).any():
        raise ValueError("Low price is above another price in the same row.")
    return cleaned.sort_values("Date").reset_index(drop=True)


datasets = {}
for company, html in pages.items():
    raw, skipped = extract_prices(html)
    datasets[company] = clean_prices(raw)
    print(f"{company}: {len(datasets[company])} price rows; {skipped} non-price rows skipped")
    display(datasets[company].head())

## 4. Compare BeautifulSoup with pandas

`read_html` is much shorter when the page already contains a well-formed table. I pass it the same selected table, remove non-price rows, and apply the same cleaning function. This checks that the two parsers produce the same dataset; it does not independently verify the accuracy of the source prices.

In [ ]:
for company, html in pages.items():
    table = find_price_table(html)
    automatic = pd.read_html(StringIO(str(table)), flavor="lxml")[0]
    automatic.columns = COLUMNS
    # Event rows repeat the notice across price columns after read_html expands colspan.
    price_rows = pd.to_datetime(
        automatic["Date"], format="%b %d, %Y", errors="coerce"
    ).notna()
    numeric_open = pd.to_numeric(
        automatic["Open"].astype(str).str.replace(",", "", regex=False), errors="coerce"
    ).notna()
    automatic = clean_prices(automatic.loc[price_rows & numeric_open])
    pd.testing.assert_frame_equal(datasets[company], automatic, check_dtype=False)
    print(f"{company}: both extraction methods agree.")

## 5. Summarize and export

The summary shows the coverage and basic closing-price range of each snapshot. These are monthly observations, so I avoid treating volume as daily volume or comparing raw share prices as a measure of company performance.

In [ ]:
summary = pd.DataFrame([
    {
        "Company": company,
        "Rows": len(frame),
        "Start": frame["Date"].min().date(),
        "End": frame["Date"].max().date(),
        "Lowest close": frame["Close"].min(),
        "Highest close": frame["Close"].max(),
    }
    for company, frame in datasets.items()
])
display(summary)

output_dir = Path("data")
output_dir.mkdir(exist_ok=True)
for company, frame in datasets.items():
    path = output_dir / f"{company.lower()}_historical_prices.csv"
    frame.to_csv(path, index=False, date_format="%Y-%m-%d")
    print(f"Saved {path}")

## What I learned and next steps

BeautifulSoup makes the row-selection logic explicit, while pandas is convenient for a regular HTML table. Most of the work after extraction is shared: converting types, deciding what to do with unusual rows, and checking the result.

This scraper depends on the snapshot's table headers and date format. It does not handle JavaScript-rendered pages, pagination, or future layout changes. A useful next step would be to add a price chart and support another table format.

## Sources

The HTML snapshots are hosted by IBM Skills Network and contain historical Yahoo Finance tables. The initial material was adapted from work by Joseph Santarcangelo, Azim Hirjani, and Akansha Yadav (IBM Corporation, 2020). The source links are defined in `SOURCES` above. This version reorganizes the workflow and adds shared extraction, cleaning, validation, and exports.